In [ ]:
# Imports
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

import os
import sys

sys.path.insert(0, os.path.abspath(".."))
from src.data.sers_io import load_dataset_with_serotype

In [ ]:
# Loading SERS Data
# --------------------------------------------------------------------------------------------------

# Define the root folder containing SERS dataset
data_folder = "../data/SERS Data 6 (Mar 2025)/"

signals_folders = ["December Signals"]

serotypes = ["ST", "SE"]

try:
    # Load the dataset using the defined function
    data_entries = load_dataset_with_serotype(data_folder, signals_folders, serotypes)

except ValueError as e:
    # Print error message if loading fails
    print(e)

In [ ]:
# Plotting data distribution
# Use a better default style
# sns.set_theme(style="whitegrid")

# Extract serotype distribution
serotypes = [entry["serotype"] for entry in data_entries]
unique_serotypes, serotype_counts = np.unique(serotypes, return_counts=True)

# Define the desired order
desired_order = ["ST", "SE"]  # Ensure these match the actual serotype names

# Create a Seaborn bar plot with the specified order
plt.figure(figsize=(6, 4))
ax = sns.barplot(
    x=unique_serotypes,
    y=serotype_counts,
    hue=unique_serotypes,
    palette="deep",
    width=0.5,
    order=desired_order,
    hue_order=desired_order,
)

# Annotate bars with count values
for p in ax.patches:
    ax.annotate(
        f"{int(p.get_height())}",
        (p.get_x() + p.get_width() / 2, p.get_height()),
        ha="center",
        va="bottom",
        fontsize=12,
        color="black",
    )

# Customize plot
plt.title("Serotype Distribution", fontsize=14, fontweight="bold")
plt.xlabel("Serotype", fontsize=12)
plt.ylabel("Count", fontsize=12)
plt.show()

In [ ]:
# Plot missing data
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib.patches as mpatches

# Extract sensor IDs, test IDs, and serotypes from the data
sensor_ids = [entry["sensor_id"] for entry in data_entries]
test_ids = [entry["test_id"] for entry in data_entries]
serotypes = [entry["serotype"] for entry in data_entries]

# Create a DataFrame to store the information
df = pd.DataFrame({"Sensor ID": sensor_ids, "Test ID": test_ids, "Serotype": serotypes})

# Determine the number of unique sensors and tests
unique_sensors = sorted(df["Sensor ID"].unique())
unique_tests = sorted(df["Test ID"].unique())

# Create a mapping of serotypes to colors using the same palette as the previous plot
serotype_palette = sns.color_palette("deep", len(df["Serotype"].unique()))
serotype_color_map = dict(zip(df["Serotype"].unique(), serotype_palette))

# Initialize the grid with NaNs
grid = np.full((len(unique_sensors), len(unique_tests)), np.nan)

# Assign colors based on the serotype
for _, row in df.iterrows():
    sensor_index = unique_sensors.index(row["Sensor ID"])
    test_index = unique_tests.index(row["Test ID"])
    grid[sensor_index, test_index] = list(serotype_color_map.keys()).index(row["Serotype"])

# Determine figure size dynamically based on sensor vs test count ratio
fig_width = len(unique_tests) * 0.5  # Scale width dynamically
fig_height = len(unique_sensors) * 0.5  # Scale height dynamically

# Create a figure with a matching aspect ratio
plt.figure(figsize=(fig_width, fig_height))

# Convert serotype color mapping to a colormap
cmap = sns.color_palette(serotype_palette, as_cmap=True)

# Plot the heatmap with square cells
sns.heatmap(grid, cmap=cmap, cbar=False, linewidths=0.5, linecolor="gray", square=True)

# Customize plot
plt.title("File Usage Grid (Sensor vs Test ID)", fontsize=14, fontweight="bold")

# Set the x-axis labels to the top
ax = plt.gca()  # Get current axis
ax.xaxis.set_tick_params(length=0)  # Hide tick marks on the x-axis
ax.xaxis.set_label_position("top")  # Move x-axis label to the top
ax.xaxis.tick_top()  # Move tick labels to the top
plt.xlabel("Test ID", fontsize=12)
plt.ylabel("Sensor ID", fontsize=12)

# Set axis ticks to align properly
plt.xticks(ticks=np.arange(len(unique_tests)) + 0.5, labels=unique_tests, rotation=0)
plt.yticks(ticks=np.arange(len(unique_sensors)) + 0.5, labels=unique_sensors, rotation=0)

# Create a legend for serotype colors
legend_patches = [
    mpatches.Patch(color=color, label=serotype) for serotype, color in serotype_color_map.items()
]
plt.legend(handles=legend_patches, title="Serotype", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.grid(False)  # Disable internal grid lines
ax.spines["top"].set_visible(True)
ax.spines["right"].set_visible(True)
ax.spines["left"].set_visible(True)
ax.spines["bottom"].set_visible(True)


# Show the plot
plt.show()

In [ ]:
# Plot data distribution
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Extract concentrations by serotype
typhimurium_conc = [entry["concentrations"] for entry in data_entries if entry["serotype"] == "ST"]
enterica_conc = [entry["concentrations"] for entry in data_entries if entry["serotype"] == "SE"]

# Flatten lists
typhimurium_conc = np.concatenate(typhimurium_conc)
enterica_conc = np.concatenate(enterica_conc)

# Define bin width
bin_width = 30  # Adjust as needed
min_conc = 0
max_conc = max(typhimurium_conc.max(), enterica_conc.max())

# Define bins based on fixed bin width
bins = np.arange(min_conc, max_conc + bin_width, bin_width)

# Define colors consistent with previous plots
colors = sns.color_palette("deep", 2)  # Ensure consistency with earlier visualizations

# Create histogram
plt.figure(figsize=(10, 5))
n, bins, patches = plt.hist(
    [typhimurium_conc, enterica_conc],
    bins=bins,
    stacked=True,
    color=colors,
    edgecolor="black",
    alpha=0.7,
    label=["ST", "SE"],
)

# Add count annotations
for i in range(len(bins) - 1):
    ty_count = n[0][i] if i < len(n[0]) else 0  # Typhimurium count in the bin
    en_count = n[1][i] - n[0][i] if i < len(n[1]) else 0  # Enterica count in the bin

    if ty_count > 0:
        plt.text(
            (bins[i] + bins[i + 1]) / 2,
            ty_count / 2,
            f"{int(ty_count)}",
            ha="center",
            va="center",
            color="black",
            fontsize=10,
        )
    if en_count > 0:
        plt.text(
            (bins[i] + bins[i + 1]) / 2,
            ty_count + en_count / 2,
            f"{int(en_count)}",
            ha="center",
            va="center",
            color="black",
            fontsize=10,
        )

# Customize x-axis labels
plt.xticks(bins, rotation=45)  # Keep in linear scale but rotated for better readability

# Add legend
plt.legend(title="Serotype")

# Customize labels and grid
plt.title("Concentration Distribution by Serotype (Linear Scale)", fontsize=14, fontweight="bold")
plt.xlabel("Concentration (CFU/ml)", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.6)

plt.show()

In [ ]:
# Plotting in concentration in log scale
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Extract concentrations by serotype
typhimurium_conc = [entry["concentrations"] for entry in data_entries if entry["serotype"] == "ST"]
enterica_conc = [entry["concentrations"] for entry in data_entries if entry["serotype"] == "SE"]

# Flatten lists
typhimurium_conc = np.concatenate(typhimurium_conc)
enterica_conc = np.concatenate(enterica_conc)

# Define bin width and log-scaled bins
# bin_width = 20
min_conc = max(1, min(typhimurium_conc.min(), enterica_conc.min()))  # Avoid log(0)
max_conc = max(typhimurium_conc.max(), enterica_conc.max())
bins = np.logspace(np.log10(min_conc), np.log10(max_conc), num=30)
# print(bins)

# Define colors consistent with previous plots
colors = sns.color_palette("deep", 2)  # Same two colors used earlier

# Create histogram
plt.figure(figsize=(10, 5))
n, bins, patches = plt.hist(
    [typhimurium_conc, enterica_conc],
    bins=bins,
    stacked=True,
    color=colors,
    edgecolor="black",
    alpha=0.7,
    label=["ST", "SE"],
)

# Set x-axis to log scale
plt.xscale("log")

# Customize x-axis labels to show log values with one decimal place
plt.xticks(bins, labels=[f"{b:.1f}" for b in bins], rotation=45)

# Add count annotations
for i in range(len(bins) - 1):
    ty_count = n[0][i] if i < len(n[0]) else 0  # Typhimurium count in the bin
    en_count = n[1][i] - n[0][i] if i < len(n[1]) else 0  # Enterica count in the bin

    # print(ty_count, en_count)
    if ty_count > 0:
        plt.text(
            (bins[i] + bins[i + 1]) / 2,
            ty_count / 2,
            f"{int(ty_count)}",
            ha="center",
            va="center",
            color="black",
            fontsize=10,
        )
    if en_count > 0:
        plt.text(
            (bins[i] + bins[i + 1]) / 2,
            ty_count + en_count / 2,
            f"{int(en_count)}",
            ha="center",
            va="center",
            color="black",
            fontsize=10,
        )

# Add legend
plt.legend(title="Serotype")

# Customize labels and grid
plt.title("Concentration Distribution by Serotype (Log10 Scale)", fontsize=14, fontweight="bold")
plt.xlabel("Concentration - Log Scale (CFU/ml)", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.6)

plt.show()

In [ ]:
# Plotting in log-transformed concentration
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Convert concentrations to log scale (log10)
typhimurium_conc_log = np.log10(
    [entry["concentrations"] for entry in data_entries if entry["serotype"] == "ST"]
)
enterica_conc_log = np.log10(
    [entry["concentrations"] for entry in data_entries if entry["serotype"] == "SE"]
)

# Flatten lists
typhimurium_conc_log = np.concatenate(typhimurium_conc_log)
enterica_conc_log = np.concatenate(enterica_conc_log)

# print(typhimurium_conc_log, enterica_conc_log)

# Define bins using log-spaced values
min_conc_log = np.min(min(typhimurium_conc_log.min(), enterica_conc_log.min()))
max_conc_log = np.max(max(typhimurium_conc_log.max(), enterica_conc_log.max()))
bins_log = np.linspace(min_conc_log, max_conc_log, num=30)  # Even spacing in log10 scale

# print(min_conc_log, max_conc_log)
# print(bins_log)

# Define colors consistent with previous plots
colors = sns.color_palette("deep", 2)

# Create histogram
plt.figure(figsize=(10, 5))  # Adjust figure size
n, bins, patches = plt.hist(
    [typhimurium_conc_log, enterica_conc_log],
    bins=bins_log,
    stacked=True,
    color=colors,
    edgecolor="black",
    alpha=0.7,
    label=["ST", "SE"],
)

# print(n)
# print(bins)


# Add count annotations
for i in range(len(bins) - 1):
    ty_count = n[0][i] if i < len(n[0]) else 0
    en_count = n[1][i] - n[0][i] if i < len(n[1]) else 0

    if ty_count > 0:
        plt.text(
            (bins[i] + bins[i + 1]) / 2,
            ty_count / 2,
            f"{int(ty_count)}",
            ha="center",
            va="center",
            color="black",
            fontsize=10,
        )
    if en_count > 0:
        plt.text(
            (bins[i] + bins[i + 1]) / 2,
            ty_count + en_count / 2,
            f"{int(en_count)}",
            ha="center",
            va="center",
            color="black",
            fontsize=10,
        )

# # Set x-axis to reflect log10 values
# plt.xticks(np.linspace(min_log, max_log, num=10), labels=[f"{b:.1f}" for b in np.linspace(min_log, max_log, num=10)], rotation=45, fontsize=10)

# # Set x-axis limits explicitly
# plt.xlim(min_log, max_log)

# Add legend
plt.legend(title="Serotype")

# Customize labels and grid
plt.title(
    "Concentration Distribution by Serotype (Log10 Transformed)", fontsize=14, fontweight="bold"
)
plt.xlabel("Log transformed Concentration (log10 CFU/ml)", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.6)

plt.show()

In [ ]:
# Showcasing 4 groups

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans

# Extract concentration values
all_concentrations = np.concatenate([entry["concentrations"] for entry in data_entries])

# Apply log transformation (Avoid log(0) error)
log_concentrations = np.log10(all_concentrations + 1e-9).reshape(-1, 1)

# Perform K-Means clustering with 4 clusters
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
clusters = kmeans.fit_predict(log_concentrations)

# Order clusters based on mean concentration
cluster_order = np.argsort(kmeans.cluster_centers_.flatten())  # Order of clusters
cluster_map = {old: new + 1 for new, old in enumerate(cluster_order)}  # Map old IDs to new
clusters = np.array([cluster_map[c] for c in clusters])  # Reassign cluster IDs

# Get cluster centers (sorted) and convert back to original scale
sorted_centers = np.sort(kmeans.cluster_centers_.flatten())
# cluster_boundaries = 10 ** sorted_centers

# Print suggested boundaries
# print("Suggested Cluster Boundaries (CFU/ml):", cluster_boundaries)

# Assign concentrations to clusters
clustered_data = {f"Cluster {i + 1}": [] for i in range(4)}
for i, conc in enumerate(all_concentrations):
    cluster_id = clusters[i]
    clustered_data[f"Cluster {cluster_id}"].append(conc)

# Define log-scaled bins
min_conc = max(1, all_concentrations.min())  # Avoid log(0)
max_conc = all_concentrations.max()
bins = np.logspace(np.log10(min_conc), np.log10(max_conc), num=30)  # Log-spaced bins

# Visualize grouped concentration ranges
plt.figure(figsize=(10, 5))
for i, (cluster, values) in enumerate(clustered_data.items()):
    sns.histplot(values, bins=bins, label=cluster, alpha=0.7)

# Set x-axis to log scale
plt.xscale("log")

# Customize x-axis labels
plt.xticks(bins, labels=[f"{b:.1f}" for b in bins], rotation=45)

# Customize plot
plt.xlabel("Concentration (CFU/ml)")
plt.ylabel("Frequency")
plt.legend(title="Clusters")
plt.title("K-Means Clustering of Concentration Values (Log Scale)")
plt.grid(axis="y", linestyle="--", alpha=0.6)

plt.show()

In [ ]:
# Visualizing Raman Shift Signal Distributions by Serotype

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Extract unique serotypes
serotypes = list(set(entry["serotype"] for entry in data_entries))

# Get consistent colors from the seaborn palette
palette = sns.color_palette("deep", len(serotypes))
serotype_colors = dict(zip(serotypes, palette))  # Map serotypes to colors

# ----- PLOT 1: Mean & Standard Deviation -----
plt.figure(figsize=(6, 4))
for serotype in serotypes:
    serotype_signals = np.array(
        [entry["signals"][:, 0] for entry in data_entries if entry["serotype"] == serotype]
    )

    mean_signal = np.mean(serotype_signals, axis=0)
    std_signal = np.std(serotype_signals, axis=0)

    color = serotype_colors[serotype]
    plt.plot(data_entries[0]["raman_shift"], mean_signal, label=f"{serotype} (Mean)", color=color)
    plt.fill_between(
        data_entries[0]["raman_shift"],
        mean_signal - std_signal,
        mean_signal + std_signal,
        color=color,
        alpha=0.2,
        label=f"{serotype} ±1 Std Dev",
    )

plt.title("Mean Raman Shift vs. Signal Intensity (Separated by Serotype)")
plt.xlabel("Raman Shift (cm⁻¹)")
plt.ylabel("Signal Intensity")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

# ----- PLOT 2: Median & Interquartile Range (IQR) -----
plt.figure(figsize=(6, 4))
for serotype in serotypes:
    serotype_signals = np.array(
        [entry["signals"][:, 0] for entry in data_entries if entry["serotype"] == serotype]
    )

    median_signal = np.median(serotype_signals, axis=0)
    q1_signal = np.percentile(serotype_signals, 25, axis=0)
    q3_signal = np.percentile(serotype_signals, 75, axis=0)

    color = serotype_colors[serotype]
    plt.plot(
        data_entries[0]["raman_shift"], median_signal, label=f"{serotype} (Median)", color=color
    )
    plt.fill_between(
        data_entries[0]["raman_shift"],
        q1_signal,
        q3_signal,
        color=color,
        alpha=0.2,
        label=f"{serotype} IQR (25-75%)",
    )

plt.title("Raman Shift vs. Signal Intensity (Median & IQR)")
plt.xlabel("Raman Shift (cm⁻¹)")
plt.ylabel("Signal Intensity")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

# ----- PLOT 3: Median & 5%-95% Range -----
plt.figure(figsize=(6, 4))
for serotype in serotypes:
    serotype_signals = np.array(
        [entry["signals"][:, 0] for entry in data_entries if entry["serotype"] == serotype]
    )

    median_signal = np.median(serotype_signals, axis=0)
    p5_signal = np.percentile(serotype_signals, 5, axis=0)
    p95_signal = np.percentile(serotype_signals, 95, axis=0)

    color = serotype_colors[serotype]
    plt.plot(
        data_entries[0]["raman_shift"], median_signal, label=f"{serotype} (Median)", color=color
    )
    plt.fill_between(
        data_entries[0]["raman_shift"],
        p5_signal,
        p95_signal,
        color=color,
        alpha=0.2,
        label=f"{serotype} 5-95%",
    )

plt.title("Raman Shift vs. Signal Intensity (Median & 5%-95% Range)")
plt.xlabel("Raman Shift (cm⁻¹)")
plt.ylabel("Signal Intensity")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

# ----- PLOT 4: Mean & Min-Max Range -----
plt.figure(figsize=(6, 4))
for serotype in serotypes:
    serotype_signals = np.array(
        [entry["signals"][:, 0] for entry in data_entries if entry["serotype"] == serotype]
    )

    mean_signal = np.mean(serotype_signals, axis=0)
    min_signal = np.min(serotype_signals, axis=0)
    max_signal = np.max(serotype_signals, axis=0)

    color = serotype_colors[serotype]
    plt.plot(data_entries[0]["raman_shift"], mean_signal, label=f"{serotype} (Mean)", color=color)
    plt.fill_between(
        data_entries[0]["raman_shift"],
        min_signal,
        max_signal,
        color=color,
        alpha=0.2,
        label=f"{serotype} Min-Max Range",
    )

plt.title("Mean Raman Shift vs. Signal Intensity (Min-Max Range by Serotype)")
plt.xlabel("Raman Shift (cm⁻¹)")
plt.ylabel("Signal Intensity")
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

In [ ]:
# Concentration-Cluster-Based Raman Shift Analysis of Serotypes

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

# Extract unique serotypes
serotypes = list(set(entry["serotype"] for entry in data_entries))

# Get consistent colors from the seaborn palette
palette = sns.color_palette("deep", len(serotypes))
serotype_colors = dict(zip(serotypes, palette))  # Map serotypes to colors

# Directly group data based on assigned clusters
grouped_data = defaultdict(list)
for entry, cluster_label in zip(data_entries, clusters):  # Use the assigned cluster labels
    grouped_data[f"Cluster {cluster_label}"].append(entry)

print("Final Grouped Data Counts:", {k: len(v) for k, v in grouped_data.items()})  # Debug

# Plot for each assigned cluster
for cluster, entries in grouped_data.items():
    plt.figure(figsize=(6, 4))  # Create a new figure
    plt.title(f"Raman Shift vs. Signal Intensity - {cluster}")

    for serotype in serotypes:
        # Filter signals by serotype within this cluster
        serotype_signals = np.array(
            [entry["signals"][:, 0] for entry in entries if entry["serotype"] == serotype]
        )

        if serotype_signals.size == 0:
            continue  # Skip if no signals for this serotype in the cluster

        # Compute median and 5%-95% range
        median_signal = np.median(serotype_signals, axis=0)
        p5_signal = np.percentile(serotype_signals, 5, axis=0)
        p95_signal = np.percentile(serotype_signals, 95, axis=0)

        # Plot median with shaded 5%-95% range
        color = serotype_colors[serotype]
        plt.plot(
            data_entries[0]["raman_shift"], median_signal, label=f"{serotype} (Median)", color=color
        )
        plt.fill_between(
            data_entries[0]["raman_shift"],
            p5_signal,
            p95_signal,
            color=color,
            alpha=0.2,
            label=f"{serotype} 5-95%",
        )

    # Customize plot
    plt.xlabel("Raman Shift (cm-1)")
    plt.ylabel("Signal Intensity")
    plt.legend()
    plt.grid(True, linestyle="--", alpha=0.6)

    plt.show()  # Show plot immediately

In [ ]:
# Plot intensity vs concentration for different serotypes (color-coded by sensors)
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Extract unique serotypes
serotypes = list(set(entry["serotype"] for entry in data_entries))

if len(serotypes) != 2:
    print(
        f"⚠️ Expected 2 serotypes but found {len(serotypes)}. Adjusting the visualization may be required."
    )

# Extract unique sensor IDs
sensor_ids = list(set(entry["sensor_id"] for entry in data_entries))

# Define colors for each sensor
colors = sns.color_palette("tab10", len(sensor_ids))
sensor_colors = dict(zip(sensor_ids, colors))

# Create figure with two subplots (left = Serotype 1, right = Serotype 2)
fig, axes = plt.subplots(1, 2, figsize=(10, 5), sharey=True)

for ax, serotype in zip(axes, serotypes):
    # Loop over each sensor
    for sensor_id in sensor_ids:
        selected_entries = [
            entry
            for entry in data_entries
            if entry["sensor_id"] == sensor_id and entry["serotype"] == serotype
        ]

        # Extract data points: max signal vs. concentration
        max_intensities = []
        concentrations = []

        for entry in selected_entries:
            for i, concentration in enumerate(entry["concentrations"]):
                max_intensity = np.max(entry["signals"][:, i])  # Get max value from the signal
                max_intensities.append(max_intensity)
                concentrations.append(concentration)

        # Convert to numpy arrays for plotting
        max_intensities = np.array(max_intensities)
        concentrations = np.array(concentrations)

        # Create scatter plot for this sensor
        ax.scatter(
            concentrations,
            max_intensities,
            alpha=0.7,
            edgecolors="black",
            color=sensor_colors[sensor_id],
            label=f"Sensor {sensor_id}",
        )

    # Customize subplot
    ax.set_xlabel("Concentration (CFU/ml)")
    ax.set_title(f"{serotype}")
    ax.set_xscale("log")  # Log scale for better visualization
    ax.grid(True, linestyle="--", alpha=0.6)
    # ax.legend(title="Sensor ID")

# Set shared Y-label
axes[0].set_ylabel("Max Signal Intensity")

# Set main title for the figure
fig.suptitle("Max Signal Intensity vs. Concentration (All Sensors)", fontsize=14, fontweight="bold")

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import math

# Extract unique sensor IDs
sensor_ids = list(set(entry["sensor_id"] for entry in data_entries))


# Function to find data entries by sensor_id
def find_entries_by_sensor(data_entries, sensor_id):
    return [entry for entry in data_entries if entry["sensor_id"] == sensor_id]


# Calculate number of rows needed (5 plots per row)
num_sensors = len(sensor_ids)
num_cols = 3  # Fixed number of columns per row
num_rows = math.ceil(num_sensors / num_cols)  # Dynamically determine rows

# Create figure with subplots
fig, axes = plt.subplots(num_rows, num_cols, figsize=(num_cols * 4, num_rows * 4), sharey=True)
axes = axes.flatten()  # Flatten to 1D array for easy indexing

# Loop over each sensor
for idx, sensor_id in enumerate(sensor_ids):
    ax = axes[idx]  # Select subplot

    # Get data for this sensor
    selected_entries = find_entries_by_sensor(data_entries, sensor_id)

    # Extract unique serotypes
    serotypes = list(set(entry["serotype"] for entry in selected_entries))

    if "ST" not in serotypes:
        print(f"⚠️ Sensor {sensor_id} does not have ST serotype. Skipping...")
        continue  # Skip sensors that do not have ST serotype

    # Extract data points: max signal vs. concentration, color by test ID
    test_id_colors = {}  # Store color mapping for test IDs
    test_ids = list(set(entry["test_id"] for entry in selected_entries))
    cmap = plt.get_cmap("tab10", len(test_ids))  # Use a colormap for different test IDs

    for entry in selected_entries:
        if entry["serotype"] != "ST":
            continue  # Only plot ST serotype

        test_id = entry["test_id"]
        if test_id not in test_id_colors:
            test_id_colors[test_id] = cmap(len(test_id_colors))  # Assign unique color

        # Extract max intensity per concentration
        max_intensities = []
        concentrations = []

        for i, concentration in enumerate(entry["concentrations"]):
            max_intensity = np.sum(entry["signals"][:, i])  # Get max value from the signal
            max_intensities.append(max_intensity)
            concentrations.append(concentration)

        # Convert to numpy arrays for plotting
        max_intensities = np.array(max_intensities)
        concentrations = np.array(concentrations)

        # Sort by concentration for smooth line connections
        sorted_indices = np.argsort(concentrations)
        concentrations = concentrations[sorted_indices]
        max_intensities = max_intensities[sorted_indices]

        # Create scatter plot for this test ID
        ax.scatter(
            concentrations,
            max_intensities,
            alpha=0.7,
            edgecolors="black",
            color=test_id_colors[test_id],
            label=f"Test {test_id}",
        )

        # Connect points with a line
        ax.plot(
            concentrations,
            max_intensities,
            linestyle="-",
            marker="",
            color=test_id_colors[test_id],
            alpha=0.6,
        )

    # Customize each subplot
    ax.set_xlabel("Concentration (CFU/ml)")
    ax.set_title(f"Sensor {sensor_id}")
    ax.set_xscale("log")  # Log scale for better visualization
    ax.grid(True, linestyle="--", alpha=0.6)

    # Show legend only for the first plot in each row
    if idx % num_cols == 0:
        ax.legend(title="Test ID", fontsize=8)

# Adjust layout and remove empty plots if any
for i in range(idx + 1, len(axes)):  # Hide unused subplots
    fig.delaxes(axes[i])

fig.suptitle(
    "Max Signal Intensity vs. Concentration for ST Serotype", fontsize=16, fontweight="bold"
)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.cm as cm


# Function to find data entries by sensor_id
def find_entries_by_sensor(data_entries, sensor_id):
    return [entry for entry in data_entries if entry["sensor_id"] == sensor_id]


# Define sensor_id
sensor_id = 1

# Find the corresponding data entries
selected_entries = find_entries_by_sensor(data_entries, sensor_id)

# Extract unique serotypes
serotypes = list(set(entry["serotype"] for entry in selected_entries))

# Loop over serotypes (Each serotype gets its own figure)
for serotype in serotypes:
    serotype_entries = [entry for entry in selected_entries if entry["serotype"] == serotype]

    # Extract all log concentrations for this serotype to determine color mapping
    all_log_concentrations = np.concatenate(
        [entry["log_concentrations"] for entry in serotype_entries]
    )
    norm = plt.Normalize(vmin=min(all_log_concentrations), vmax=max(all_log_concentrations))
    cmap = cm.autumn  # Continuous colormap

    # Create figure
    fig, ax = plt.subplots(figsize=(10, 10))

    for entry in serotype_entries:
        test_id = entry["test_id"]
        for i, log_concentration in enumerate(entry["log_concentrations"]):
            color = cmap(norm(log_concentration))  # Assign color based on log concentration
            ax.plot(
                entry["raman_shift"],
                entry["signals"][:, i],
                color=color,
                label=f"Test {test_id}, log10(CFU/ml) {log_concentration:.2f}",
            )

    # Customize plot
    ax.set_title(f"Signal Visualization for Sensor {sensor_id} - {serotype}")
    ax.set_xlabel("Raman Shift (cm⁻¹)")
    ax.set_ylabel("Signal Intensity")
    ax.grid(True, linestyle="--", alpha=0.6)

    # Place colorbar on the right of the plot
    sm = cm.ScalarMappable(norm=norm, cmap=cmap)
    cbar = fig.colorbar(sm, ax=ax, orientation="vertical", pad=0.02)
    cbar.set_label("Log10 Concentration (CFU/ml)")

    # Show legend
    ax.legend(loc="upper right")

    plt.tight_layout()
    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.cm as cm


# Function to find data entries by sensor_id
def find_entries_by_sensor(data_entries, sensor_id):
    return [entry for entry in data_entries if entry["sensor_id"] == sensor_id]


# Define sensor_id
sensor_id = 1

# Find the corresponding data entries
selected_entries = find_entries_by_sensor(data_entries, sensor_id)

# Extract unique serotypes
serotypes = list(set(entry["serotype"] for entry in selected_entries))

# Loop over serotypes (Each serotype gets its own figure per test ID)
for serotype in serotypes:
    serotype_entries = [entry for entry in selected_entries if entry["serotype"] == serotype]

    # Extract unique test IDs for this serotype
    test_ids = list(set(entry["test_id"] for entry in serotype_entries))

    for test_id in test_ids:
        # Filter entries for the current test_id
        test_entries = [entry for entry in serotype_entries if entry["test_id"] == test_id]

        # Extract all log concentrations for this serotype & test_id to determine color mapping
        all_log_concentrations = np.concatenate(
            [entry["log_concentrations"] for entry in test_entries]
        )
        norm = plt.Normalize(vmin=min(all_log_concentrations), vmax=max(all_log_concentrations))
        cmap = cm.autumn  # Continuous colormap

        # Create figure
        fig, ax = plt.subplots(figsize=(10, 6))

        for entry in test_entries:
            for i, log_concentration in enumerate(entry["log_concentrations"]):
                color = cmap(norm(log_concentration))  # Assign color based on log concentration
                ax.plot(
                    entry["raman_shift"],
                    entry["signals"][:, i],
                    color=color,
                    label=f"log10(CFU/ml) {log_concentration:.2f}",
                )

        # Customize plot
        ax.set_title(f"Sensor {sensor_id} - {serotype} (Test {test_id})")
        ax.set_xlabel("Raman Shift (cm⁻¹)")
        ax.set_ylabel("Signal Intensity")
        ax.grid(True, linestyle="--", alpha=0.6)

        # Place colorbar on the right of the plot
        sm = cm.ScalarMappable(norm=norm, cmap=cmap)
        cbar = fig.colorbar(sm, ax=ax, orientation="vertical", pad=0.02)
        cbar.set_label("Log10 Concentration (CFU/ml)")

        # Show legend on the right
        ax.legend(loc="upper right")

        plt.tight_layout()
        plt.show()

In [ ]:
import matplotlib.pyplot as plt


# Function to find data entries by sensor_id and test_id
def find_entries_by_sensor_and_test(data_entries, sensor_id, test_ids):
    selected_entries = []
    for entry in data_entries:
        if entry["sensor_id"] == sensor_id and entry["test_id"] in test_ids:
            selected_entries.append(entry)
    return selected_entries


# Define sensor_id and test_ids
sensor_id = 2
test_ids = [1, 3]

# Find the corresponding data entries
selected_entries = find_entries_by_sensor_and_test(data_entries, sensor_id, test_ids)

# Check if the entries exist
if len(selected_entries) < len(test_ids):
    raise ValueError(
        f"Not all requested test IDs ({test_ids}) were found for sensor_id {sensor_id}."
    )

# Plot signals for the selected entries
plt.figure(figsize=(10, 6))

for entry in selected_entries:
    test_id = entry["test_id"]
    for i, concentration in enumerate(entry["concentrations"]):
        plt.plot(
            entry["raman_shift"],
            entry["signals"][:, i],
            label=f"Sensor {sensor_id}, Test {test_id}, {concentration} CFU/ml",
        )

# Customize plot
plt.title("Signal Visualization for Sensor 2 and Test IDs 1, 2")
plt.xlabel("Raman Shift")
plt.ylabel("Signal Intensity")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import defaultdict

# Organize data by sensor
sensor_data = defaultdict(list)
for entry in data_entries:
    sensor_data[entry["sensor_id"]].append(entry)

# Set up a consistent color palette for serotypes
serotypes = list(set(entry["serotype"] for entry in data_entries))
palette = sns.color_palette("deep", len(serotypes))
serotype_colors = dict(zip(serotypes, palette))

# Plot all signals for each sensor
for sensor_id, entries in sensor_data.items():
    plt.figure(figsize=(12, 6))
    plt.title(f"Raman Shift vs. Signal Intensity - Sensor {sensor_id}")

    for entry in entries:
        raman_shift = entry["raman_shift"]
        signals = entry["signals"]
        serotype = entry["serotype"]
        color = serotype_colors.get(serotype, "gray")  # Default to gray if missing

        # Plot all signals for this sensor
        for i in range(signals.shape[1]):  # Iterate over all concentration levels
            plt.plot(raman_shift, signals[:, i], alpha=0.5, color=color)

    # Customize plot
    plt.xlabel("Raman Shift (cm⁻¹)")
    plt.ylabel("Signal Intensity")
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.show()

In [ ]:
import torch
from torch.utils.data import Dataset
from sklearn.preprocessing import LabelEncoder


class SERSDataset(Dataset):
    def __init__(self, data_entries, transform=None):
        """
        A PyTorch-compatible dataset for SERS data.

        Args:
            data_entries (list of dict): Each dict contains:
                - 'signals': Signal intensities (2D array with shape [shift_length, num_concentrations]).
                - 'serotype': Serotype label (str).
                - 'concentrations': List of concentration values (1D array).
            transform (callable, optional): A function/transform to apply to the signals for augmentation.
        """
        self.transform = transform
        self.data = []
        self.labels = []

        # Label encoder for serotype
        self.serotype_encoder = LabelEncoder()
        serotypes = [entry["serotype"] for entry in data_entries]
        self.serotype_encoder.fit(serotypes)

        for entry in data_entries:
            for i, concentration in enumerate(entry["concentrations"]):
                # Skip signals with concentration == 0
                if concentration == 0:
                    continue

                # Extract the signal for this concentration
                signal = torch.tensor(entry["signals"][:, i], dtype=torch.float32)

                # Apply optional transformations
                if self.transform:
                    signal = self.transform(signal)

                # Encode serotype and create labels
                serotype_label = torch.tensor(
                    self.serotype_encoder.transform([entry["serotype"]])[0], dtype=torch.long
                )
                concentration_label = torch.tensor(float(concentration), dtype=torch.float32)

                # Append to dataset
                self.data.append(signal)
                self.labels.append((serotype_label, concentration_label))

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        return self.data[idx], self.labels[idx]

In [ ]:
def add_noise(signal, noise_level=0.01):
    noise = torch.randn_like(signal) * noise_level
    return signal + noise

In [ ]:
# Do sensor specefic normalization for ST

# Define the selected sensors
selected_sensors = [3, 4, 5, 8, 9, 10, 11, 12, 16, 17, 19]

# Filter data_entries to include only ST serotype and selected sensors
filtered_entries = [
    entry
    for entry in data_entries
    if entry["serotype"] == "ST" and entry["sensor_id"] in selected_sensors
]

# Display filtered dataset size
print("Length of original entries", len(data_entries))
print(
    f"✅ Filtered dataset contains {len(filtered_entries)} entries after applying serotype and sensor selection."
)

In [ ]:
import numpy as np

# Step 1: Define Target CFU Ranges (Mapping CFU values to standard categories)
target_CFU_ranges = {
    "1 CFU": (0.5, 5),  # CFU between 0.5 and 5
    "10 CFU": (5, 50),  # CFU between 5 and 50
    "100 CFU": (50, 500),  # CFU between 50 and 500
    "1000 CFU": (500, float("inf")),  # CFU above 500
}

# Step 2: Modify `data_entries` to Include a Target CFU Key
for entry in filtered_entries:
    concentrations = entry["concentrations"]

    # Assign each concentration to its corresponding target CFU range
    target_labels = []
    for cfu in concentrations:
        for target, (low, high) in target_CFU_ranges.items():
            if low <= cfu <= high:
                target_labels.append(target)
                break  # Assign to the first matching range and move on

    entry["target_CFU"] = target_labels  # Add new key to `data_entries`

# Step 3: Compute Mean Signal for Each Target CFU for Each Sensor
sensor_signals_by_target = {
    sensor_id: {key: [] for key in target_CFU_ranges} for sensor_id in selected_sensors
}

for entry in filtered_entries:
    sensor_id = entry["sensor_id"]
    signals = entry["signals"]  # Shape: (raman_shift_length, num_concentrations)
    target_CFU_labels = entry["target_CFU"]  # Assigned CFU labels for each concentration

    for i, target in enumerate(target_CFU_labels):
        sensor_signals_by_target[sensor_id][target].append(
            signals[:, i]
        )  # Store the signal under correct CFU target

# Step 4: Compute Mean Signal Per CFU Range for Each Sensor
sensor_mean_signals = {sensor_id: {} for sensor_id in selected_sensors}

for sensor_id in selected_sensors:
    for target in target_CFU_ranges:
        if sensor_signals_by_target[sensor_id][target]:  # Ensure there is data
            sensor_mean_signals[sensor_id][target] = np.mean(
                sensor_signals_by_target[sensor_id][target], axis=0
            )

# Step 5: Compute Max Intensity Per CFU Range for Each Sensor
sensor_max_intensities = {sensor_id: {} for sensor_id in selected_sensors}

for sensor_id in selected_sensors:
    for target in target_CFU_ranges:
        if target in sensor_mean_signals[sensor_id]:
            sensor_max_intensities[sensor_id][target] = np.max(
                sensor_mean_signals[sensor_id][target]
            )

# # Step 6: Select the First Sensor as the Reference
# reference_sensor = selected_sensors[0]
# sensor_scaling_factors = {sensor_id: {} for sensor_id in selected_sensors}

# for target in target_CFU_ranges:
#     if target in sensor_max_intensities[reference_sensor]:
#         reference_intensity = sensor_max_intensities[reference_sensor][target]

#         for sensor_id in selected_sensors:
#             if target in sensor_max_intensities[sensor_id]:
#                 sensor_intensity = sensor_max_intensities[sensor_id][target]
#                 scaling_factor = reference_intensity / sensor_intensity if sensor_intensity > 0 else 1.0
#                 sensor_scaling_factors[sensor_id][target] = scaling_factor

# Step 6: Manually Set Scaling Factors for Each Target CFU
sensor_scaling_factors = {sensor_id: {} for sensor_id in selected_sensors}

for sensor_id in selected_sensors:
    sensor_scaling_factors[sensor_id]["1 CFU"] = 1.0  # No change
    sensor_scaling_factors[sensor_id]["10 CFU"] = 2.0  # Scale by 2
    sensor_scaling_factors[sensor_id]["100 CFU"] = 3.0  # Scale by 3
    sensor_scaling_factors[sensor_id]["1000 CFU"] = 4.0  # Scale by 4

# Step 7: Print Normalization Coefficients for Each Sensor
print("\n✅ Normalization Coefficients:")
for sensor_id in selected_sensors:
    print(f"\nSensor {sensor_id}:")
    for target in target_CFU_ranges:
        factor = sensor_scaling_factors[sensor_id].get(target, 1.0)
        print(f"  {target}: Scaling Factor = {factor:.4f}")

# Step 8: Apply Scaling to All Signals Based on Their CFU Range
for entry in filtered_entries:
    sensor_id = entry["sensor_id"]
    signals = entry["signals"]
    target_CFU_labels = entry["target_CFU"]

    for i, target in enumerate(target_CFU_labels):
        if target in sensor_scaling_factors[sensor_id]:
            scaling_factor = sensor_scaling_factors[sensor_id][target]
            entry["signals"][:, i] *= scaling_factor  # Apply normalization

print("\n✅ Normalization applied to all signals with 4 separate scaling factors per sensor.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

data_entries = filtered_entries

# Extract unique sensor IDs
sensor_ids = list(set(entry["sensor_id"] for entry in data_entries))


# Function to find data entries by sensor_id
def find_entries_by_sensor(data_entries, sensor_id):
    return [entry for entry in data_entries if entry["sensor_id"] == sensor_id]


# Loop over each sensor
for sensor_id in sensor_ids:
    # Get data for this sensor
    selected_entries = find_entries_by_sensor(data_entries, sensor_id)

    # Extract unique serotypes
    serotypes = list(set(entry["serotype"] for entry in selected_entries))

    if len(serotypes) != 2:
        print(f"⚠️ Sensor {sensor_id} has {len(serotypes)} serotypes, expected 2. Skipping...")
        # continue  # Ensure exactly 2 serotypes per sensor for side-by-side plots

    # Create figure with two subplots (left = serotype 1, right = serotype 2)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

    for ax, serotype in zip(axes, serotypes):
        serotype_entries = [entry for entry in selected_entries if entry["serotype"] == serotype]

        # Extract data points: max signal vs. concentration
        max_intensities = []
        concentrations = []

        for entry in serotype_entries:
            for i, concentration in enumerate(entry["concentrations"]):
                max_intensity = np.max(entry["signals"][:, i])  # Get max value from the signal
                max_intensities.append(max_intensity)
                concentrations.append(concentration)

        # Convert to numpy arrays for plotting
        max_intensities = np.array(max_intensities)
        concentrations = np.array(concentrations)

        # Create scatter plot
        ax.scatter(concentrations, max_intensities, alpha=0.7, edgecolors="black")

        # Customize subplot
        ax.set_xlabel("Concentration (CFU/ml)")
        ax.set_title(f"{serotype} (Sensor {sensor_id})")
        ax.set_xscale("log")  # Log scale for better visualization
        ax.grid(True, linestyle="--", alpha=0.6)

    # Set shared Y-label
    axes[0].set_ylabel("Max Signal Intensity")

    # Set main title for the sensor figure
    fig.suptitle(
        f"Max Signal Intensity vs. Concentration (Sensor {sensor_id})",
        fontsize=14,
        fontweight="bold",
    )

    plt.tight_layout()
    plt.show()

In [ ]:
from torch.utils.data import DataLoader

# Assuming `data_entries` is the list of dictionaries loaded from the data files
# Initialize the dataset
dataset = SERSDataset(data_entries)

# Split the dataset into training and testing sets
train_size = int(0.8 * len(dataset))  # 80% training
test_size = len(dataset) - train_size  # 20% testing
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

# Calculate and display the sizes of train and test sets
print(f"Training Set Size: {train_size}")
print(f"Testing Set Size: {test_size}")

# Create DataLoader objects for batching
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=len(test_dataset), shuffle=False)

# Example: Print a batch of data to verify
for batch_idx, (features, labels) in enumerate(train_loader):
    print(f"Batch {batch_idx + 1}:")
    print(f"Features: {features.shape}")  # Shape: [batch_size, feature_length]
    print(f"Labels: {labels}")  # Each label is a tuple (serotype_label, concentration_label)
    break  # Only show the first batch

In [ ]:
from torch.utils.data import DataLoader

# Filter `data_entries` to include only `ST` (Typhimurium) serotype
st_data_entries = [entry for entry in data_entries if entry["serotype"] == "ST"]

# Ensure we still have data after filtering
if not st_data_entries:
    raise ValueError("No ST samples found in the dataset!")

# Initialize the dataset with only ST data
dataset = SERSDataset(st_data_entries)

# Split the dataset into training and testing sets (80% training, 20% testing)
train_size = int(0.8 * len(dataset))
test_size = len(dataset) - train_size
train_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, test_size])

# Display the sizes of train and test sets
print(f"Training Set Size (ST only): {train_size}")
print(f"Testing Set Size (ST only): {test_size}")

# Create DataLoader objects for batching
train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=len(test_dataset), shuffle=False)

# Example: Print a batch of data to verify
for batch_idx, (features, labels) in enumerate(train_loader):
    print(f"Batch {batch_idx + 1}:")
    print(f"Features: {features.shape}")  # Shape: [batch_size, feature_length]
    print(f"Labels: {labels}")  # Each label is a tuple (serotype_label, concentration_label)
    break  # Only show the first batch

In [ ]:
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import torch

# Extract signals and labels from the dataset
signals = []
labels = []

for i in range(len(dataset)):
    signal, label = dataset[i]
    signals.append(signal.numpy())  # Convert torch tensor to numpy
    labels.append(label[0].item())  # Extract serotype_label (first element of label tuple)

# Convert lists to numpy arrays for PCA
signals = np.array(signals)
labels = np.array(labels)

# Perform PCA to reduce to 2 dimensions
pca = PCA(n_components=2)
reduced_data = pca.fit_transform(signals)

# Plot PCA results
plt.figure(figsize=(10, 6))

# Assign colors to different labels
unique_labels = np.unique(labels)
for label in unique_labels:
    idx = labels == label
    plt.scatter(reduced_data[idx, 0], reduced_data[idx, 1], label=f"Serotype {label}", alpha=0.7)

# Customize plot
plt.title("PCA Visualization of SERS Dataset")
plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import torch.nn as nn

# class MultiTaskNN(nn.Module):
#     def __init__(self, input_size, num_classes):
#         super(MultiTaskNN, self).__init__()
#         self.shared_layer = nn.Sequential(
#             nn.Linear(input_size, 128),
#             nn.ReLU(),
#             nn.Linear(128, 64),
#             nn.ReLU()
#         )

#         # Classification head for serotype
#         self.classifier = nn.Linear(64, num_classes)

#         # Regression head for concentration
#         self.regressor = nn.Linear(64, 1)

#     def forward(self, x):
#         shared_features = self.shared_layer(x)

#         # Classification output
#         class_out = self.classifier(shared_features)

#         # Regression output
#         reg_out = self.regressor(shared_features)

#         return class_out, reg_out


class ClassificationNN(nn.Module):
    def __init__(self, input_size, num_classes):
        super(ClassificationNN, self).__init__()
        self.layers = nn.Sequential(
            nn.Linear(input_size, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes),
            # nn.Sigmoid()
        )

    def forward(self, x):
        return self.layers(x)


class ImprovedRegressionNN(nn.Module):
    def __init__(self, input_size):
        super(ImprovedRegressionNN, self).__init__()

        self.fc1 = nn.Linear(input_size, 128)
        self.bn1 = nn.BatchNorm1d(128)  # Batch Normalization
        self.fc2 = nn.Linear(128, 256)
        self.bn2 = nn.BatchNorm1d(256)
        self.fc3 = nn.Linear(256, 128)
        self.bn3 = nn.BatchNorm1d(128)
        self.fc4 = nn.Linear(128, 64)
        self.bn4 = nn.BatchNorm1d(64)
        self.fc5 = nn.Linear(64, 1)  # Final output layer

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(0.2)  # Slightly higher dropout to prevent overfitting

    def forward(self, x):
        x = self.relu(self.bn1(self.fc1(x)))
        x = self.dropout(x)
        x = self.relu(self.bn2(self.fc2(x)))
        x = self.dropout(x)
        x = self.relu(self.bn3(self.fc3(x)))  # Fixed missing activation
        x = self.dropout(x)
        x = self.relu(self.bn4(self.fc4(x)))  # Fixed missing activation
        x = self.fc5(x)  # Ensure output layer is applied
        return x

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# Model setup
input_size = dataset[0][0].shape[0]  # Input feature size from the dataset
num_classes = len(set([entry["serotype"] for entry in data_entries]))  # Number of unique serotypes

# Initialize the classification model, optimizer, and loss function
classification_model = ClassificationNN(input_size=input_size, num_classes=num_classes)
optimizer = optim.Adam(classification_model.parameters(), lr=0.001, weight_decay=1e-3)
criterion_class = nn.CrossEntropyLoss()

# Tracking losses for plotting
losses = {"train_class_loss": [], "test_class_loss": []}

# Training setup
num_epochs = 200
best_test_loss = float("inf")  # Track the best test loss
best_epoch = -1
best_model_state = None

for epoch in range(num_epochs):
    # Training phase
    classification_model.train()
    total_class_loss = 0

    for x, label in train_loader:  # Only serotype_label is used
        optimizer.zero_grad()

        # Forward pass
        class_out = classification_model(x)

        # Calculate classification loss
        loss_class = criterion_class(class_out, label[0])  # Use serotype_label
        loss_class.backward()

        # Update parameters
        optimizer.step()

        # Accumulate losses for reporting
        total_class_loss += loss_class.item()

    # Calculate average training loss
    avg_train_class_loss = total_class_loss / len(train_loader)
    losses["train_class_loss"].append(avg_train_class_loss)

    # Testing phase
    classification_model.eval()
    total_test_class_loss = 0

    ground_truth = []  # Store ground truth labels
    predictions = []  # Store predicted labels

    with torch.no_grad():
        for x, label in test_loader:  # Only serotype_label is used
            # Forward pass
            class_out = classification_model(x)

            # Calculate classification loss
            test_loss_class = criterion_class(class_out, label[0])  # Use serotype_label

            # Accumulate test losses
            total_test_class_loss += test_loss_class.item()

            # Get predictions (argmax over logits)
            predicted_labels = torch.argmax(class_out, dim=1)

            # Store ground truth and predictions
            ground_truth.extend(label[0].tolist())  # Convert tensor to list
            predictions.extend(predicted_labels.tolist())

    # Calculate average test loss
    avg_test_class_loss = total_test_class_loss / len(test_loader)
    losses["test_class_loss"].append(avg_test_class_loss)

    # Track the best model
    if avg_test_class_loss < best_test_loss:
        best_test_loss = avg_test_class_loss
        best_epoch = epoch
        best_model_state = classification_model.state_dict()

    # Print loss for each epoch
    print(
        f"Epoch [{epoch + 1}/{num_epochs}], "
        f"Train Classification Loss: {avg_train_class_loss:.4f} | "
        f"Test Classification Loss: {avg_test_class_loss:.4f}"
    )

# Save the trained classification model
torch.save(classification_model.state_dict(), "classification_model.pth")
print("Classification model saved successfully!")

# Load the best model and evaluate final predictions
classification_model.load_state_dict(best_model_state)
classification_model.eval()

# Final ground truth vs. prediction and accuracy calculation
correct = sum(1 for gt, pred in zip(ground_truth, predictions) if gt == pred)
accuracy = correct / len(ground_truth)

print(f"Best model loaded from epoch {best_epoch + 1} with test loss {best_test_loss:.4f}")
print("Ground Truth Labels: ", ground_truth)
print("Predicted Labels:    ", predictions)
print(f"Accuracy: {accuracy * 100:.2f}%")

# Plot the classification losses
plt.figure(figsize=(10, 6))
plt.plot(losses["train_class_loss"], label="Train Classification Loss", linewidth=2)
plt.plot(losses["test_class_loss"], label="Test Classification Loss", linestyle="--", linewidth=2)
plt.title("Training and Testing Classification Loss Over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay


# Define class labels (ensure they are in the correct order)
class_labels = ["Ty", "En"]  # Replace with actual class names

# Compute the confusion matrix
cm = confusion_matrix(ground_truth, predictions)

# Display the confusion matrix
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_labels)
disp.plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.show()

In [ ]:
correct_count = sum(1 for gt, pred in zip(ground_truth, predictions) if gt == pred)
incorrect_count = len(ground_truth) - correct_count

plt.figure(figsize=(6, 6))
plt.bar(["Correct", "Incorrect"], [correct_count, incorrect_count], color=["green", "red"])
plt.title("Correct vs Incorrect Predictions")
plt.ylabel("Count")
plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# Model setup
input_size = dataset[0][0].shape[0]  # Input feature size from the dataset

# Initialize the regression model, optimizer, and loss function
regression_model = ImprovedRegressionNN(input_size=input_size)  # Assuming RegressionNN is defined
optimizer = optim.Adam(regression_model.parameters(), lr=0.001, weight_decay=1e-4)

# **Learning rate scheduler**
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=300, eta_min=1e-5)

criterion_reg = nn.MSELoss()

# Tracking losses and best model details
losses = {"train_reg_loss": [], "test_reg_loss": []}

num_epochs = 500
best_test_loss = float("inf")  # Track the best test loss
best_epoch = -1
best_model_state = None
best_ground_truth = None  # Best epoch's ground truth values
best_predictions = None  # Best epoch's predictions

for epoch in range(num_epochs):
    # Training phase
    regression_model.train()
    total_reg_loss = 0

    for x, label in train_loader:  # Only concentration_label is used
        optimizer.zero_grad()

        # Forward pass
        reg_out = regression_model(x)

        # Calculate regression loss
        loss_reg = criterion_reg(reg_out.squeeze(), label[1])  # Use concentration_label
        loss_reg.backward()

        # Update parameters
        optimizer.step()

        # Accumulate losses for reporting
        total_reg_loss += loss_reg.item()

    # Step the learning rate scheduler
    scheduler.step()

    # Calculate average training loss
    avg_train_reg_loss = total_reg_loss / len(train_loader)
    losses["train_reg_loss"].append(avg_train_reg_loss)

    # Testing phase
    regression_model.eval()
    total_test_reg_loss = 0

    ground_truth = []  # Store ground truth concentrations
    predictions = []  # Store predicted concentrations

    with torch.no_grad():
        for x, label in test_loader:  # Only concentration_label is used
            # Forward pass
            reg_out = regression_model(x)

            # Calculate regression loss
            test_loss_reg = criterion_reg(reg_out.squeeze(), label[1])  # Use concentration_label

            # Accumulate test losses
            total_test_reg_loss += test_loss_reg.item()

            # Store ground truth and predictions for best epoch tracking
            ground_truth.extend(label[1].tolist())  # Convert tensor to list
            predictions.extend(reg_out.squeeze().tolist())  # Convert tensor to list

    # Calculate average test loss
    avg_test_reg_loss = total_test_reg_loss / len(test_loader)
    losses["test_reg_loss"].append(avg_test_reg_loss)

    # Track the best model (lowest test loss)
    if avg_test_reg_loss < best_test_loss:
        best_test_loss = avg_test_reg_loss
        best_epoch = epoch
        best_model_state = regression_model.state_dict()
        best_ground_truth = ground_truth.copy()  # Save ground truth at best epoch
        best_predictions = predictions.copy()  # Save predictions at best epoch

    # **Print formatted epoch information**
    print(
        f"Epoch [{epoch + 1:03d}/{num_epochs}], "
        f"Train Regression Loss: {avg_train_reg_loss:.4f} | "
        f"Test Regression Loss: {avg_test_reg_loss:.4f} | "
        f"LR: {scheduler.get_last_lr()[0]:.6f}"
    )

# Save the trained regression model
torch.save(regression_model.state_dict(), "regression_model.pth")
print("✅ Regression model saved successfully!")

# Load the best model and evaluate final predictions
regression_model.load_state_dict(best_model_state)
regression_model.eval()

# Compute MAE & MSE based on the **best model's predictions**
mae = sum(abs(gt - pred) for gt, pred in zip(best_ground_truth, best_predictions)) / len(
    best_ground_truth
)
mse = sum((gt - pred) ** 2 for gt, pred in zip(best_ground_truth, best_predictions)) / len(
    best_ground_truth
)

print(f"\n🎯 Best model loaded from epoch {best_epoch + 1} with test loss {best_test_loss:.4f}")
print(f"📉 Best Test Regression Loss: {best_test_loss:.4f}")
print(f"📊 Mean Absolute Error (Best Model): {mae:.4f}")
print(f"📊 Mean Squared Error (Best Model): {mse:.4f}")

# Plot the regression losses
plt.figure(figsize=(10, 6))
plt.plot(losses["train_reg_loss"], label="Train Regression Loss", linewidth=2)
plt.plot(losses["test_reg_loss"], label="Test Regression Loss", linestyle="--", linewidth=2)
plt.title("Training and Testing Regression Loss Over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Train & test on only ST concentration

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# Filter dataset to include only ST (Typhimurium) samples
st_train_data = [(x, label) for x, label in train_dataset if label[0] == "ST"]
st_test_data = [(x, label) for x, label in test_dataset if label[0] == "ST"]

# Ensure we still have data
if not st_train_data or not st_test_data:
    raise ValueError("No ST samples found in train or test dataset!")

# Create new data loaders for ST only
train_loader = DataLoader(st_train_data, batch_size=32, shuffle=True)
test_loader = DataLoader(st_test_data, batch_size=32, shuffle=False)

# Model setup
input_size = st_train_data[0][0].shape[0]  # Input feature size from dataset

# Initialize the regression model, optimizer, and loss function
regression_model = ImprovedRegressionNN(input_size=input_size)  # Assuming RegressionNN is defined
optimizer = optim.Adam(regression_model.parameters(), lr=0.001, weight_decay=1e-4)
criterion_reg = nn.MSELoss()

# Tracking losses for plotting
losses = {"train_reg_loss": [], "test_reg_loss": []}

# Training setup
num_epochs = 500
best_test_loss = float("inf")  # Track the best test loss
best_epoch = -1
best_model_state = None

for epoch in range(num_epochs):
    # Training phase
    regression_model.train()
    total_reg_loss = 0

    for x, label in train_loader:
        optimizer.zero_grad()

        # Forward pass
        reg_out = regression_model(x)

        # Calculate regression loss
        loss_reg = criterion_reg(reg_out.squeeze(), label[1])  # Use concentration_label
        loss_reg.backward()

        # Update parameters
        optimizer.step()

        # Accumulate losses for reporting
        total_reg_loss += loss_reg.item()

    # Calculate average training loss
    avg_train_reg_loss = total_reg_loss / len(train_loader)
    losses["train_reg_loss"].append(avg_train_reg_loss)

    # Testing phase
    regression_model.eval()
    total_test_reg_loss = 0

    ground_truth = []  # Store ground truth concentrations
    predictions = []  # Store predicted concentrations

    with torch.no_grad():
        for x, label in test_loader:
            # Forward pass
            reg_out = regression_model(x)

            # Calculate regression loss
            test_loss_reg = criterion_reg(reg_out.squeeze(), label[1])  # Use concentration_label

            # Accumulate test losses
            total_test_reg_loss += test_loss_reg.item()

            # Store ground truth and predictions
            ground_truth.extend(label[1].tolist())  # Convert tensor to list
            predictions.extend(reg_out.squeeze().tolist())  # Convert tensor to list

    # Calculate average test loss
    avg_test_reg_loss = total_test_reg_loss / len(test_loader)
    losses["test_reg_loss"].append(avg_test_reg_loss)

    # Track the best model
    if avg_test_reg_loss < best_test_loss:
        best_test_loss = avg_test_reg_loss
        best_epoch = epoch
        best_model_state = regression_model.state_dict()

    # Print loss for each epoch
    print(
        f"Epoch [{epoch + 1}/{num_epochs}], "
        f"Train Regression Loss: {avg_train_reg_loss:.4f} | "
        f"Test Regression Loss: {avg_test_reg_loss:.4f}"
    )

# Save the trained regression model
torch.save(regression_model.state_dict(), "regression_model_ST.pth")
print("Regression model (ST only) saved successfully!")

# Load the best model and evaluate final predictions
regression_model.load_state_dict(best_model_state)
regression_model.eval()

# Final ground truth vs. prediction and metrics calculation
mae = sum(abs(gt - pred) for gt, pred in zip(ground_truth, predictions)) / len(ground_truth)
mse = sum((gt - pred) ** 2 for gt, pred in zip(ground_truth, predictions)) / len(ground_truth)

print(f"Best model loaded from epoch {best_epoch + 1} with test loss {best_test_loss:.4f}")
print("Ground Truth Concentrations: ", ground_truth)
print("Predicted Concentrations:    ", predictions)
print(f"Mean Absolute Error: {mae:.4f}")
print(f"Mean Squared Error: {mse:.4f}")

# Plot the regression losses
plt.figure(figsize=(10, 6))
plt.plot(losses["train_reg_loss"], label="Train Regression Loss", linewidth=2)
plt.plot(losses["test_reg_loss"], label="Test Regression Loss", linestyle="--", linewidth=2)
plt.title("Training and Testing Regression Loss Over Epochs (ST Only)")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Plot Total Loss
plt.figure(figsize=(10, 6))
plt.plot(losses["train_total_loss"], label="Train Total Loss", linewidth=2)
plt.plot(losses["test_total_loss"], label="Test Total Loss", linestyle="--", linewidth=2)
plt.title("Total Loss Over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Plot Classification Loss
plt.figure(figsize=(10, 6))
plt.plot(losses["train_class_loss"], label="Train Classification Loss", linewidth=2)
plt.plot(losses["test_class_loss"], label="Test Classification Loss", linestyle="--", linewidth=2)
plt.title("Classification Loss Over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

# Plot Regression Loss
plt.figure(figsize=(10, 6))
plt.plot(losses["train_reg_loss"], label="Train Regression Loss", linewidth=2)
plt.plot(losses["test_reg_loss"], label="Test Regression Loss", linestyle="--", linewidth=2)
plt.title("Regression Loss Over Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# Calculate absolute errors
absolute_errors = [abs(pred - gt) for pred, gt in zip(predictions, ground_truth)]

# Plot histogram of absolute errors
plt.figure(figsize=(8, 6))
plt.hist(absolute_errors, bins=20, color="skyblue", edgecolor="black", alpha=0.7)
plt.xlabel("Absolute Error")
plt.ylabel("Frequency")
plt.title("Distribution of Absolute Errors")
plt.grid(axis="y", linestyle="--", linewidth=0.5)
plt.show()

In [ ]:
# Scatter plot
plt.figure(figsize=(8, 6))
plt.scatter(ground_truth, predictions, alpha=0.6, label="Predictions")
plt.plot(
    [min(ground_truth), max(ground_truth)],
    [min(ground_truth), max(ground_truth)],
    "r--",
    label="Ideal Fit",
)
plt.xlabel("Ground Truth Concentration")
plt.ylabel("Predicted Concentration")
plt.title("Ground Truth vs. Predicted Concentrations")
plt.legend()
plt.grid(True, which="both", linestyle="--", linewidth=0.5)
plt.show()